In [ ]:
from google.colab import userdata
import os
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('langsmith')
os.environ["LANGCHAIN_PROJECT"] = "Wedding Planner"

In [ ]:
!pip install -U langchain-google-genai -q
from langchain_google_genai import ChatGoogleGenerativeAI


In [ ]:
API_key = userdata.get('gemini')

llm_model = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] =  API_key
model = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)


# Setup Tools

In [ ]:
!pip install -U langchain-mcp-adapters -q
from langchain_mcp_adapters.client import MultiServerMCPClient

Client = MultiServerMCPClient(
  {
    "travel_server": {
    "transport": "streamable_http",
    "url": "https://mcp.kiwi.com"
}
}
)

tools = await Client.get_tools()

In [ ]:
! pip install tavily -q
from typing import Dict, Any
from tavily import TavilyClient
from langchain. tools import tool

tavily = TavilyClient(api_key=userdata.get('tavily'))

@tool
def web_search(query: str) -> Dict[str, Any]:

  """Search the web for information"""

  return tavily.search(query)

In [ ]:
!pip install -U langchain-community -q

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase. from_uri("sqlite:///Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
      return db. run(query)
    except Exception as e:
      return f"Error querying database: {e}"

# Create State

In [ ]:
from langchain.agents import AgentState

In [ ]:
class WeddingState(AgentState):
  origin: str
  destination: str
  guest_count: str
  genre: str

# Create Subagents

In [ ]:
from langchain.agents import create_agent

travel_agent = create_agent(
    model = model,
    tools = tools,
    system_prompt="""
            You are a travel agent. Search for flights to the desired destination wedding location.,
            You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:,
            - Price (lowest, economy class),
            - Duration (shortest),
            - Date (time of year which you believe is best for a wedding at this location),
            To make things easy, only look for one ticket, one way.,
            You may need to make multiple searches to iteratively find the best options.,
            You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.,
            If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.,
            Once you have found the best options, let the user know your shortlist of options.
    """
)

In [ ]:
venue_agent = create_agent(
    model = model,
    tools = [web_search],
        system_prompt="""
            You are a venue specialist. Search for venues in the desired location, and with the desired capacity.,
            You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:,
            - Price (lowest),
            - Capacity (exact match),
            - Reviews (highest),
            You may need to make multiple searches to iteratively find the best options.,
            You have a suggested limit of 12 web searches. Count every web_search call you make.,
            After 12 searches, you should stop searching and summarize the best options you have,
            found so far.
            """
)

In [ ]:
playlist_agent = create_agent(
    model = model,
    tools = [query_playlist_db],
    system_prompt= """
        You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.,
        Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.,
        If you run into errors when querying the database, try to fix them by making changes to the query.,
        Do not come back empty handed, keep trying to query the db until you find a list of songs.,
          This is a SQLite database. Before writing any data queries, first discover the schema.,
    """
    )

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command


@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Search for flights to the wedding destination."""

    origin = runtime.state["origin"]
    destination = runtime.state["destination"]

    query = f"Find flights from {origin} to {destination}"

    response = await travel_agent.ainvoke({
        "messages": [HumanMessage(content=query)]
    })

    return response["messages"][-1].content


@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Find the best wedding venue for the location and guest capacity."""

    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]

    query = f"Find wedding venues in {destination} for {capacity} guests"

    response = venue_agent.invoke({
        "messages": [HumanMessage(content=query)]
    })

    return response["messages"][-1].content


@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Suggest a music playlist for the wedding."""

    genre = runtime.state["genre"]

    query = f"Find {genre} tracks suitable for a wedding playlist"

    response = playlist_agent.invoke({
        "messages": [HumanMessage(content=query)]
    })

    return response["messages"][-1].content

In [ ]:
@tool
def update_state(origin: str,destination: str,guest_count: str,genre: str,runtime: ToolRuntime) -> Command:
    """
    Update the shared state once all required values are known:
    origin, destination, guest_count, and genre.
    """

    return Command(update={
        "origin": origin,
        "destination": destination,
        "guest_count": guest_count,
        "genre": genre,
        "messages": [
            ToolMessage(
                content="Successfully updated state",
                tool_call_id=runtime.tool_call_id
            )
        ]
    })

In [ ]:
from langchain.agents import create_agent

coordinator = create_agent (
  model=model,
  tools=[search_flights, search_venues, suggest_playlist, update_state],
  state_schema=WeddingState,
  system_prompt="""
    You are a wedding coordinator. Delegate tasks to your specialists for flights, venues and playlists.
    First find all the information you need to update the state. Once that is done you can delegate the tasks.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)


# Test

In [ ]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
  {
  "messages": [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre")]
  }
)